In [23]:
!pip install scikit-learn


Defaulting to user installation because normal site-packages is not writeable


In [24]:
import pandas as pd
import numpy as np
import sklearn.feature_extraction.text as text
from sklearn.metrics.pairwise import cosine_similarity
vectorizer = text.TfidfVectorizer()

In [25]:
dataset = pd.read_csv('../data/amazon_products.csv',nrows=10000)
print('=== Dataset Loaded ===')
print(f"Shape: {dataset.shape}")
print(f"Columns: {dataset.columns.tolist()}")

=== Dataset Loaded ===
Shape: (10000, 11)
Columns: ['asin', 'title', 'imgUrl', 'productURL', 'stars', 'reviews', 'price', 'listPrice', 'category_id', 'isBestSeller', 'boughtInLastMonth']


In [26]:
# creating the product descrption from the columns
def create_product_text(row):
    """Combine product features into one text string"""
    features = []
    # Adding title
    features.append(str(row['title']))

    # Adding Category
    features.append(f"category_{row['category_id']}")

    # Add price range (helps find similarly priced products)
    if row['price'] > 0:
        if row['price'] < 50:
            features.append("budget")
        elif row['price'] < 200:
            features.append("mid_range")
        else:
            features.append("Premium")

    
    # Adding Bestseller info
    if(row['isBestSeller']):
        features.append("bestSeller")

    return " ".join(features)

# Applying to all the products
dataset['product_text'] = dataset.apply(create_product_text, axis=1)
print('Product features Created')
print("\nExample:")
print(dataset['product_text'].iloc[0])


Product features Created

Example:
Sion Softside Expandable Roller Luggage, Black, Checked-Large 29-Inch category_104 mid_range


In [27]:
# Converting text to numerical numbers
vectorizer = text.TfidfVectorizer(stop_words='english', max_features=5000)
tfidf_matrix = vectorizer.fit_transform(dataset['product_text'])
print(f"TF-IDF matrix created")
print(f"Shape: {tfidf_matrix.shape}")
print(f"Each Product is now a {tfidf_matrix.shape[1]}-dimensional vector")

TF-IDF matrix created
Shape: (10000, 5000)
Each Product is now a 5000-dimensional vector


In [28]:
# Calualating the cosine similarities between all the products
cosine_sim = cosine_similarity(tfidf_matrix,tfidf_matrix)
print(f"Similarity matrix created")
print(f"Shape: {cosine_sim.shape}")
print(f"Similarity between  prosuct 0 and 1: {cosine_sim[0][1]:.3f}")

Similarity matrix created
Shape: (10000, 10000)
Similarity between  prosuct 0 and 1: 0.154
